# S&P 500 Options: PatchTST

This notebook fits the declared PatchTST member of the sequence population snapshotted by
`09_deep_learning`. After publishing every PatchTST checkpoint, it verifies that the complete
NLinear, LSTM, and PatchTST population is present.

Prerequisites: `09_deep_learning` and `09a_lstm`.

In [1]:
"""Fit the declared S&P 500 options PatchTST request."""

import polars as pl

from case_studies.sp500_options.research_workflow import (
    ALL_LABELS,
    declared_dl_device,
    model_request_catalog,
    open_study,
    published_dl_device,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_subset,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}
DEVICE: str = ""

POPULATION_NAME: str = ""

### The device the population was fitted on

A network trained on a GPU and the same network trained on a CPU accumulate their sums in a
different order and reach different weights, so the device is part of what the fitted model is
and sits inside the training identity rather than beside it. The device this population was
fitted on is declared once, in `modeling.dl.device` in `config/setup.yaml`, and read from there
by all four deep-learning notebooks rather than retyped in each. On a machine with no NVIDIA
card the run stops here rather than quietly training something else: set `DEVICE="cpu"` and pass
a `POPULATION_NAME` to fit the same requests there, under a name of their own.

In [3]:
CANONICAL_POPULATION_NAME = "sp500-options-sequence-validation-v1"

published_device = published_dl_device()
device = declared_dl_device(DEVICE)
population_name = POPULATION_NAME or CANONICAL_POPULATION_NAME
if device != published_device and population_name == CANONICAL_POPULATION_NAME:
    raise ValueError(
        f"this run fits on {device!r}, not the published {published_device!r}, so its "
        f"identities are not the ones {CANONICAL_POPULATION_NAME!r} holds; pass "
        f"POPULATION_NAME to give them a population of their own"
    )
print(f"training device: {device} (declared: {published_device})")

training device: cuda (declared: cuda)


## Declared request

In [4]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
requests = model_request_catalog(
    "deep_learning",
    labels=ALL_LABELS,
    config_names=("patchtst",),
)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    overrides={"device": device},
    preview_reductions=PREVIEW_REDUCTIONS,
)
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""deep_learning""","""ret_to_expiry""","""patchtst""","""regression""",52,248,42186,2,2019-01-07 00:00:00,2020-11-10 00:00:00,20,"""canonical""","""ee9423e378aa"""


## Execute and validate

The shared sequence runner owns gap-safe window construction, fold fitting, fitted-state reload,
checkpoint publication, restart, and exact eligible-key validation.

In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_subset(
        study,
        resolved,
        population=population_name,
        require_population_complete=True,
    )
else:
    if not WORKSPACE or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("PatchTST execution returned a partial checkpoint")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",5,"""canonical""",true,"""ee9423e378aa""","""00e65965e8be"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",10,"""canonical""",true,"""ee9423e378aa""","""8f8c52686d51"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",15,"""canonical""",true,"""ee9423e378aa""","""6545c006daca"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",20,"""canonical""",true,"""ee9423e378aa""","""799abafe9271"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",25,"""canonical""",true,"""ee9423e378aa""","""8976ae8a0ef9"""
…,…,…,…,…,…,…,…,…
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",80,"""canonical""",true,"""ee9423e378aa""","""1c865f22d3ca"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",85,"""canonical""",true,"""ee9423e378aa""","""272c5b7188e0"""
"""deep_learning""","""ret_to_expiry""","""patchtst""","""epoch""",90,"""canonical""",true,"""ee9423e378aa""","""fcb0c80d5490"""


The official sequence population is complete and ready for model analysis and backtesting. This
notebook does not compare configurations or choose a checkpoint.